In [1]:
"""Legacy entry point for the local tariff RAG flow using the Ollama prompt template."""

## src/main.py
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from src.config import OLLAMA_MODEL
from src.ingestion import initialize_vector_store
from src.extraction import PromptExtractor
from src.retriever import TariffRetriever
#from src.vessel import vessel_profile


def run_pipeline(user_query: str):
    print("Initializing Vector Database...")
    vector_store = initialize_vector_store()
    
    print("Extracting parameters from prompt...")
    extractor = PromptExtractor()
    extracted_entities = extractor.extract(user_query)
    print(f"Extracted Entities: {extracted_entities}")
    
    print("Retrieving relevant tariff rules...")
    retriever = TariffRetriever(vector_store)
    context = retriever.get_relevant_context(user_query)
    
    print("Generating calculations and response via Local LLM...")

    # llm = Ollama(model=OLLAMA_MODEL)
    

    # connect to LM Studio server for LLM
    llm = ChatOpenAI(
        base_url="http://127.0.0.1:1234/v1",  # LM Studio server endpoint
        api_key="lm-studio",
        model="meta-llama-3.1-8b-instruct",    #"google/gemma-4-12b",  # Match the exact model name shown in LM Studio     
        temperature=0.1
)
    rag_prompt = PromptTemplate.from_template(
        """" Calculate the different tariffs payable by the following vessel berthing at the port of <Durban | Saldanha | Richard's Bay>:


        Context Clauses:
        {context}

        User Query: {user_query}
        Extracted Variables: {entities}

        Provide a structured, step-by-step breakdown:"""
    )
    
    final_prompt = rag_prompt.format(
        context=context,
        user_query=user_query,
        entities=extracted_entities
    )
    
    response = llm.invoke(final_prompt)
    return response

if __name__ == "__main__":
    sample_query = "Calculate the different tariffs payable by the following vessel berthing at the port of <Durban | Saldanha | Richard's Bay>"
    
    result = run_pipeline(sample_query)
    print("\n=== Tariff Calculation Output ===\n")
    print(result)

ModuleNotFoundError: No module named 'src'